In [ ]:
# imports

from pathlib import Path
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
# paths and constants

PROCESSED_DIR = Path("processed_data")
RESULTS_DIR = Path("results") / "final_models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"
STATIC_FILE = PROCESSED_DIR / "penmanshiel_static.parquet"

TEST_START = pd.Timestamp("2022-01-01", tz="UTC")
TEST_END = pd.Timestamp("2023-01-01", tz="UTC")

SEEDS = [1, 21, 42, 84, 123]

N_TURBINES = 14

BATCH_SIZE = 64
MAX_EPOCHS = 300
PATIENCE = 30

HIDDEN_A = (64, 32)
HIDDEN_B = (64, 32)
HIDDEN_C = (64, 32)

LEARNING_RATE_A = 5e-4
LEARNING_RATE_B = 1e-3
LEARNING_RATE_C = 1e-3

POWER_CURVE_BIN_WIDTH = 0.5
POWER_CURVE_WS_MAX = 30.0


In [ ]:
# reproducibility

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


In [ ]:
# build the common farm-level modelling table

def build_modelling_data(scada):
    data = scada.copy()

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True,
    )

    turbine_ids = sorted(
        data["turbine_id"].astype(int).unique()
    )

    power_wide = (
        data
        .pivot(
            index="timestamp",
            columns="turbine_id",
            values="power_kw",
        )
        .reindex(columns=turbine_ids)
        .sort_index()
    )

    wind_speed_wide = (
        data
        .pivot(
            index="timestamp",
            columns="turbine_id",
            values="wind_speed",
        )
        .reindex(columns=turbine_ids)
        .sort_index()
    )

    wind_direction_wide = (
        data
        .pivot(
            index="timestamp",
            columns="turbine_id",
            values="wind_dir",
        )
        .reindex(columns=turbine_ids)
        .sort_index()
    )

    direction_radians = np.deg2rad(
        wind_direction_wide.to_numpy(dtype=float)
    )

    mean_sine = np.nanmean(
        np.sin(direction_radians),
        axis=1,
    )

    mean_cosine = np.nanmean(
        np.cos(direction_radians),
        axis=1,
    )

    farm_data = pd.DataFrame(
        index=power_wide.index
    )

    farm_data["global_ws"] = wind_speed_wide.max(axis=1)
    farm_data["wd_sin"] = mean_sine
    farm_data["wd_cos"] = mean_cosine
    farm_data["total_power"] = power_wide.sum(axis=1)

    for turbine_id in turbine_ids:
        farm_data[f"ws_t{turbine_id}"] = (
            wind_speed_wide[turbine_id]
        )

        farm_data[f"power_t{turbine_id}"] = (
            power_wide[turbine_id]
        )

    farm_data = farm_data.dropna().sort_index()

    return farm_data, turbine_ids


In [ ]:
# chronological train, validation and test split

def chronological_split(
    farm_data,
    test_start=TEST_START,
    test_end=TEST_END,
):
    test_data = farm_data.loc[
        (farm_data.index >= test_start)
        & (farm_data.index < test_end)
    ].copy()

    pre_test_data = farm_data.loc[
        farm_data.index < test_start
    ].copy()

    split_index = int(
        len(pre_test_data) * 0.9
    )

    train_data = pre_test_data.iloc[
        :split_index
    ].copy()

    validation_data = pre_test_data.iloc[
        split_index:
    ].copy()

    if (
        train_data.empty
        or validation_data.empty
        or test_data.empty
    ):
        raise ValueError(
            "training, validation and test periods must all contain data"
        )

    return train_data, validation_data, test_data


In [ ]:
# training-only standardisation

def fit_standardisation(values):
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    mean = values.mean(axis=0)
    std = values.std(axis=0, ddof=1)

    std = np.where(
        std == 0,
        1.0,
        std,
    )

    return mean, std


def standardise(values, mean, std):
    return (
        np.asarray(values, dtype=np.float32)
        - mean
    ) / std


In [ ]:
# feed-forward model

def build_network(
    input_size,
    output_size,
    hidden_layers,
):
    return nn.Sequential(
        nn.Linear(
            input_size,
            hidden_layers[0],
        ),
        nn.ReLU(),
        nn.Linear(
            hidden_layers[0],
            hidden_layers[1],
        ),
        nn.ReLU(),
        nn.Linear(
            hidden_layers[1],
            output_size,
        ),
    )


In [ ]:
# train one neural network with early stopping

def train_network(
    X_train,
    y_train,
    X_validation,
    y_validation,
    hidden_layers,
    learning_rate,
    seed,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
):
    set_seed(seed)

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.float32,
    )

    X_validation_tensor = torch.tensor(
        X_validation,
        dtype=torch.float32,
    )

    y_validation_tensor = torch.tensor(
        y_validation,
        dtype=torch.float32,
    )

    model = build_network(
        input_size=X_train_tensor.shape[1],
        output_size=y_train_tensor.shape[1],
        hidden_layers=hidden_layers,
    )

    loss_function = nn.HuberLoss(
        delta=1.0
    )

    optimiser = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    training_loader = DataLoader(
        TensorDataset(
            X_train_tensor,
            y_train_tensor,
        ),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )

    history = {
        "train": [],
        "validation": [],
    }

    best_validation_loss = np.inf
    best_state = None
    best_epoch = None
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()

        for X_batch, y_batch in training_loader:
            optimiser.zero_grad()

            predictions = model(X_batch)
            loss = loss_function(
                predictions,
                y_batch,
            )

            loss.backward()
            optimiser.step()

        model.eval()

        with torch.no_grad():
            training_loss = loss_function(
                model(X_train_tensor),
                y_train_tensor,
            ).item()

            validation_loss = loss_function(
                model(X_validation_tensor),
                y_validation_tensor,
            ).item()

        history["train"].append(
            training_loss
        )

        history["validation"].append(
            validation_loss
        )

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch + 1
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    if best_state is None:
        raise RuntimeError(
            "no best model state was saved"
        )

    model.load_state_dict(best_state)
    model.eval()

    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_validation_loss": best_validation_loss,
    }


In [ ]:
# empirical turbine power curves for model B

def build_power_curves(
    scada,
    training_timestamps,
    turbine_ids,
    bin_width=POWER_CURVE_BIN_WIDTH,
    ws_max=POWER_CURVE_WS_MAX,
):
    data = scada.copy()

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True,
    )

    training_timestamps = pd.DatetimeIndex(
        training_timestamps
    )

    data = data[
        data["timestamp"].isin(
            training_timestamps
        )
    ].copy()

    bin_edges = np.arange(
        0,
        ws_max + bin_width,
        bin_width,
    )

    curves = {}

    for turbine_id in turbine_ids:
        turbine_data = data[
            data["turbine_id"] == turbine_id
        ].copy()

        turbine_data["wind_speed_bin"] = pd.cut(
            turbine_data["wind_speed"],
            bins=bin_edges,
            include_lowest=True,
        )

        median_power = (
            turbine_data
            .groupby(
                "wind_speed_bin",
                observed=True,
            )["power_kw"]
            .median()
            .dropna()
        )

        wind_speed_centres = np.array(
            [
                interval.mid
                for interval in median_power.index
            ],
            dtype=float,
        )

        curves[turbine_id] = {
            "wind_speed": wind_speed_centres,
            "power_kw": median_power.to_numpy(
                dtype=float
            ),
        }

    return curves


def speeds_to_turbine_power(
    wind_speeds,
    curves,
    turbine_ids,
):
    wind_speeds = np.asarray(
        wind_speeds,
        dtype=float,
    )

    turbine_power = np.zeros_like(
        wind_speeds,
        dtype=float,
    )

    for column_index, turbine_id in enumerate(
        turbine_ids
    ):
        curve = curves[turbine_id]

        turbine_power[:, column_index] = np.interp(
            np.clip(
                wind_speeds[:, column_index],
                0,
                POWER_CURVE_WS_MAX,
            ),
            curve["wind_speed"],
            curve["power_kw"],
            left=0.0,
            right=curve["power_kw"][-1],
        )

    return turbine_power


In [ ]:
# common regression metrics

def regression_metrics(
    actual,
    predicted,
):
    actual = np.asarray(
        actual,
        dtype=float,
    )

    predicted = np.asarray(
        predicted,
        dtype=float,
    )

    residual = predicted - actual

    return {
        "MAE_kW": mean_absolute_error(
            actual,
            predicted,
        ),
        "RMSE_kW": np.sqrt(
            mean_squared_error(
                actual,
                predicted,
            )
        ),
        "R2": r2_score(
            actual,
            predicted,
        ),
        "bias_kW": residual.mean(),
    }


In [ ]:
# run models A, B and C for one seed

def run_models_ABC(
    farm_data,
    scada,
    static,
    turbine_ids,
    seed,
):
    feature_columns = [
        "global_ws",
        "wd_sin",
        "wd_cos",
    ]

    wind_speed_columns = [
        f"ws_t{turbine_id}"
        for turbine_id in turbine_ids
    ]

    turbine_power_columns = [
        f"power_t{turbine_id}"
        for turbine_id in turbine_ids
    ]

    train_data, validation_data, test_data = (
        chronological_split(
            farm_data
        )
    )

    farm_capacity_kw = static.loc[
        static["turbine_id"].isin(
            turbine_ids
        ),
        "rated_power_kw",
    ].sum()

    X_train_raw = train_data[
        feature_columns
    ].to_numpy()

    X_validation_raw = validation_data[
        feature_columns
    ].to_numpy()

    X_test_raw = test_data[
        feature_columns
    ].to_numpy()

    X_mean, X_std = fit_standardisation(
        X_train_raw
    )

    X_train = standardise(
        X_train_raw,
        X_mean,
        X_std,
    )

    X_validation = standardise(
        X_validation_raw,
        X_mean,
        X_std,
    )

    X_test = standardise(
        X_test_raw,
        X_mean,
        X_std,
    )

    actual_farm_power = test_data[
        "total_power"
    ].to_numpy()

    # linear regression baseline

    linear_model = LinearRegression().fit(
        X_train_raw,
        train_data["total_power"].to_numpy(),
    )

    linear_prediction = np.clip(
        linear_model.predict(
            X_test_raw
        ),
        0,
        farm_capacity_kw,
    )

    # model A

    y_A_train_raw = train_data[
        ["total_power"]
    ].to_numpy()

    y_A_validation_raw = validation_data[
        ["total_power"]
    ].to_numpy()

    y_A_mean, y_A_std = fit_standardisation(
        y_A_train_raw
    )

    y_A_train = standardise(
        y_A_train_raw,
        y_A_mean,
        y_A_std,
    )

    y_A_validation = standardise(
        y_A_validation_raw,
        y_A_mean,
        y_A_std,
    )

    result_A = train_network(
        X_train=X_train,
        y_train=y_A_train,
        X_validation=X_validation,
        y_validation=y_A_validation,
        hidden_layers=HIDDEN_A,
        learning_rate=LEARNING_RATE_A,
        seed=seed,
    )

    with torch.no_grad():
        pred_A_standardised = (
            result_A["model"](
                torch.tensor(
                    X_test,
                    dtype=torch.float32,
                )
            )
            .numpy()
        )

    pred_A_power = (
        pred_A_standardised
        * y_A_std
        + y_A_mean
    ).reshape(-1)

    pred_A_power = np.clip(
        pred_A_power,
        0,
        farm_capacity_kw,
    )

    # model B

    y_B_train_raw = train_data[
        wind_speed_columns
    ].to_numpy()

    y_B_validation_raw = validation_data[
        wind_speed_columns
    ].to_numpy()

    y_B_mean, y_B_std = fit_standardisation(
        y_B_train_raw
    )

    y_B_train = standardise(
        y_B_train_raw,
        y_B_mean,
        y_B_std,
    )

    y_B_validation = standardise(
        y_B_validation_raw,
        y_B_mean,
        y_B_std,
    )

    result_B = train_network(
        X_train=X_train,
        y_train=y_B_train,
        X_validation=X_validation,
        y_validation=y_B_validation,
        hidden_layers=HIDDEN_B,
        learning_rate=LEARNING_RATE_B,
        seed=seed,
    )

    with torch.no_grad():
        pred_B_ws_standardised = (
            result_B["model"](
                torch.tensor(
                    X_test,
                    dtype=torch.float32,
                )
            )
            .numpy()
        )

    pred_B_ws = (
        pred_B_ws_standardised
        * y_B_std
        + y_B_mean
    )

    actual_B_ws = test_data[
        wind_speed_columns
    ].to_numpy()

    power_curves = build_power_curves(
        scada=scada,
        training_timestamps=train_data.index,
        turbine_ids=turbine_ids,
    )

    pred_B_turbine_power = (
        speeds_to_turbine_power(
            pred_B_ws,
            power_curves,
            turbine_ids,
        )
    )

    oracle_turbine_power = (
        speeds_to_turbine_power(
            actual_B_ws,
            power_curves,
            turbine_ids,
        )
    )

    pred_B_power = np.clip(
        pred_B_turbine_power.sum(axis=1),
        0,
        farm_capacity_kw,
    )

    oracle_power = np.clip(
        oracle_turbine_power.sum(axis=1),
        0,
        farm_capacity_kw,
    )

    # model C

    y_C_train_raw = train_data[
        turbine_power_columns
    ].to_numpy()

    y_C_validation_raw = validation_data[
        turbine_power_columns
    ].to_numpy()

    y_C_mean, y_C_std = fit_standardisation(
        y_C_train_raw
    )

    y_C_train = standardise(
        y_C_train_raw,
        y_C_mean,
        y_C_std,
    )

    y_C_validation = standardise(
        y_C_validation_raw,
        y_C_mean,
        y_C_std,
    )

    result_C = train_network(
        X_train=X_train,
        y_train=y_C_train,
        X_validation=X_validation,
        y_validation=y_C_validation,
        hidden_layers=HIDDEN_C,
        learning_rate=LEARNING_RATE_C,
        seed=seed,
    )

    with torch.no_grad():
        pred_C_standardised = (
            result_C["model"](
                torch.tensor(
                    X_test,
                    dtype=torch.float32,
                )
            )
            .numpy()
        )

    pred_C_turbine_power = (
        pred_C_standardised
        * y_C_std
        + y_C_mean
    )

    pred_C_power = np.clip(
        pred_C_turbine_power.sum(axis=1),
        0,
        farm_capacity_kw,
    )

    actual_C_turbine_power = test_data[
        turbine_power_columns
    ].to_numpy()

    # metrics

    metrics = {
        "Linear regression": regression_metrics(
            actual_farm_power,
            linear_prediction,
        ),
        "Model A": regression_metrics(
            actual_farm_power,
            pred_A_power,
        ),
        "Model B": regression_metrics(
            actual_farm_power,
            pred_B_power,
        ),
        "Model C": regression_metrics(
            actual_farm_power,
            pred_C_power,
        ),
        "Measured-speed reference": regression_metrics(
            actual_farm_power,
            oracle_power,
        ),
    }

    B_ws_metrics = {
        "MAE_ms": mean_absolute_error(
            actual_B_ws.reshape(-1),
            pred_B_ws.reshape(-1),
        ),
        "R2": r2_score(
            actual_B_ws.reshape(-1),
            pred_B_ws.reshape(-1),
        ),
    }

    return {
        "seed": seed,
        "train_index": train_data.index,
        "validation_index": validation_data.index,
        "test_index": test_data.index,
        "test_data": test_data[
            feature_columns
        ].copy(),
        "turbine_ids": turbine_ids,
        "actual_farm_power": actual_farm_power,
        "linear_prediction": linear_prediction,
        "pred_A_power": pred_A_power,
        "pred_B_power": pred_B_power,
        "pred_C_power": pred_C_power,
        "oracle_power": oracle_power,
        "actual_B_ws": actual_B_ws,
        "pred_B_ws": pred_B_ws,
        "actual_C_turbine_power": actual_C_turbine_power,
        "pred_B_turbine_power": pred_B_turbine_power,
        "pred_C_turbine_power": pred_C_turbine_power,
        "metrics": metrics,
        "B_ws_metrics": B_ws_metrics,
        "best_epochs": {
            "Model A": result_A["best_epoch"],
            "Model B": result_B["best_epoch"],
            "Model C": result_C["best_epoch"],
        },
        "best_validation_losses": {
            "Model A": result_A[
                "best_validation_loss"
            ],
            "Model B": result_B[
                "best_validation_loss"
            ],
            "Model C": result_C[
                "best_validation_loss"
            ],
        },
        "loss_history": {
            "Model A": result_A["history"],
            "Model B": result_B["history"],
            "Model C": result_C["history"],
        },
        "power_curves": power_curves,
        "normalisation": {
            "X_mean": X_mean,
            "X_std": X_std,
            "A_mean": y_A_mean,
            "A_std": y_A_std,
            "B_mean": y_B_mean,
            "B_std": y_B_std,
            "C_mean": y_C_mean,
            "C_std": y_C_std,
        },
    }
